# Street View Insights (`SV_PANO`): Blind Corridor House Discovery, 3D Multi-View Triangulation & Verification

[Open in Google Colab][colab] · [Open in Colab Enterprise][colab-ent] · [View on GitHub][github]

This notebook provides an end-to-end, self-contained reference implementation for **discovering, triangulating, and verifying buildings and roadside assets directly from raw Street View Insights (`SV_PANO`) camera frames**—without needing any prior building coordinates or address list as input.

### What This Notebook Covers

| Stage | Module / Step | What It Does |
| :--- | :--- | :--- |
| **1. Blind Image Selection** | BigQuery (`pano_observations_latest`) | Sweeps an `80 m` street corridor (`~9–11` sequential car captures × `6` horizontal rosette cameras `_0.._5` = `~60` frames) using `ST_DWITHIN` spatial pruning. |
| **2. Feature Extraction** | Vertex AI Gemini (`google-genai`) | Extracts structured building features in each frame: `facade_box_2d`, `front_door_box_2d`, `left/right_facade_corner_u`, `building_category`, `stories`, and `visible_house_number` (OCR). |
| **3. 3D View Grouping & Triangulation** | Calibrated Pinhole + Huber IRLS | Converts unlabeled 2D pixel keypoints `(u, v)` + 6-DoF camera poses into 3D compass rays, groups views of the same building via **unsupervised 3D sightline clustering** (with OCR & same-frame **Cannot-Link** guards), triangulates the **3D front entrance**, and shifts inward perpendicular to the facade plane (`+5.8 m`) to predict the **rooftop centroid**. |
| **4. Quality Gates & Verification** | Overture Maps + Google Places API v1 | Applies automated quality gates (`ray_rms_m <= 0.50 m`, `n_door_views >= 1`, `n_views >= 3`), runs **1-to-1 Hungarian bipartite matching** against `bigquery-public-data.overture_maps.building` and **Google Places API v1 (`ChIJ`)**, and explains how to interpret **F1, Precision, Recall, Ray RMS, and In-Roof Hit Rate**. |

> **Zero External Helper Dependencies:** This notebook uses only standard Google Cloud & scientific Python packages (`google-cloud-bigquery`, `google-genai`, `numpy`, `scipy`, `pandas`, `shapely`, `Pillow`, `matplotlib`) pre-installed in Google Colab and Vertex AI Colab Enterprise.

[colab]: https://colab.research.google.com/github/googlemaps-samples/insights-samples/blob/svi-gemini-improvements/street_view_insights/panoramic/notebooks/blind_corridor_house_triangulation_and_verification.ipynb
[colab-ent]: https://console.cloud.google.com/vertex-ai/colab/import
[github]: https://github.com/googlemaps-samples/insights-samples/blob/svi-gemini-improvements/street_view_insights/panoramic/notebooks/blind_corridor_house_triangulation_and_verification.ipynb

## 1. Terms of Service, Attribution & IAM Prerequisites

* **Terms of Use & Attribution:** Imagery comes from your **Google Maps Platform Street View Insights (`SV_PANO`)** dataset and is governed by your Google Cloud / Google Maps Platform agreement. Always display the `"Imagery © Google"` attribution on rendered figures and maps, and stream `gcs_uri` images in memory rather than persisting bulk imagery or `gcs_uri` strings beyond short-lived working caches (`<= 35 days` snapshot retention TTL).
* **Required GCP IAM Roles:**
  * `roles/bigquery.jobUser` (to run BigQuery queries in your billing project)
  * `roles/bigquery.dataViewer` (on your subscribed `imagery_insights___us` / `imagery_insights___eu` dataset)
  * `roles/storage.objectViewer` (on the backing Street View Insights Cloud Storage bucket)
  * `roles/aiplatform.user` (to call Vertex AI Gemini models)

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# @title Step 1: Install & Import Standard Scientific / Google Cloud Libraries
import dataclasses
from dataclasses import dataclass, field
import io
import json
import math
import os
import sys
import urllib.parse
from typing import Any, Mapping, Sequence

import google.auth
from google.auth.transport.requests import AuthorizedSession
from google.cloud import bigquery
from google import genai
from google.genai import types
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from PIL import Image
from pydantic import BaseModel, Field
from scipy.optimize import linear_sum_assignment
from shapely.geometry import Point, Polygon as ShapelyPolygon
from shapely import wkt

# Authenticate automatically if running inside Google Colab
if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()

print("All required libraries imported successfully.")

## 2. Configure Project, Street Corridor & Model Parameters

We configure:
1. **`PROJECT_ID` & `SVI_DATASET`**: Your Google Cloud project ID and subscribed Street View Insights BigQuery dataset (`imagery-insights-sandbox.imagery_insights___us` by default).
2. **`CORRIDOR_LAT` & `CORRIDOR_LNG`**: The center point of an **`80 m` residential street segment** (`W Myrtle St, Lakeland, FL 33815`). Notice that we only specify the **street corridor center**—we do **not** pass any individual building coordinates or addresses into the model.
3. **`GEMINI_MODEL`**: The Vertex AI Gemini model used for per-frame building & door detection (`gemini-2.5-flash` by default for customer projects).

In [ ]:
# @title Step 2: Configure GCP Project, Corridor Location & Gemini Model
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "imagery-insights-sandbox")  # @param {type:"string"}
SVI_PROJECT_ID = "imagery-insights-sandbox"  # @param {type:"string"}
SVI_DATASET = "imagery_insights___us"  # @param {type:"string"}
VERTEX_LOCATION = "global"  # @param {type:"string"}
GEMINI_MODEL = "gemini-2.5-flash"  # @param {type:"string"}

# Center of our 80m street corridor (W Myrtle St, Lakeland, FL 33815)
CORRIDOR_LAT = 28.05070  # @param {type:"number"}
CORRIDOR_LNG = -81.96126  # @param {type:"number"}
CORRIDOR_RADIUS_M = 45.0  # @param {type:"number"}

# Optional: Google Places API v1 key or OAuth project for post-hoc ChIJ verification
ENABLE_GOOGLE_PLACES_VERIFICATION = True  # @param {type:"boolean"}

bq_client = bigquery.Client(project=PROJECT_ID)
creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
authed_http = AuthorizedSession(creds)

print(f"Configured BigQuery client (billing project: {PROJECT_ID})")
print(f"Target 80m corridor center: ({CORRIDOR_LAT:.5f}, {CORRIDOR_LNG:.5f}), radius={CORRIDOR_RADIUS_M}m")

## 3. Step 1 of Pipeline: How We Select Street View Images (Blind Corridor Sweep)

### Why We Sweep the Corridor Instead of Looking Up Addresses
In real-world asset discovery (e.g., finding unmapped residences, commercial storefronts, utility poles, or street trees), you don't know where every target is beforehand. Instead, we sweep an **`80 m` street corridor**:

1. **Query `pano_observations_latest` (`SV_PANO`) with `ST_DWITHIN` First**:
   * Always query the `*_latest` view (never `*_all` or expiring snapshot tables) so every `gcs_uri` is active within the 35-day retention window.
   * Filter with `ST_DWITHIN` **before** consuming `gcs_uri` so BigQuery prunes candidate rows prior to joining the underlying `urls_<snapshot>` table.
2. **Select Only Horizontal Rosette Cameras (`_0` to `_5`)**:
   * Each `capture_id` (`t1:<pano_id>`) has 7 cameras triggered at the exact same microsecond (`0 µs` intra-capture spread) with `observation_id` formatted as `o1:<pano_id>_<cam_k>:5001ee`: cameras `_0` through `_5` form a **360° horizontal ring** (`60°` spacing, `3648 × 5472 px` portrait frames), while camera `_6` points straight up at the sky (zenith). We filter to `CAST(REGEXP_EXTRACT(observation_id, r'_(\d+)(?::|$)') AS INT64) BETWEEN 0 AND 5`.
3. **Single-Drive Pass Filtering (Optional Best Practice)**:
   * Because `SV_PANO` pools historical Street View collects into an S2 cell priority queue with `>= 5.0 m` 3D spacing, a street corridor can contain frames from multiple drive dates. Below we select the primary continuous car pass (or top `MAX_PANOS = 10` captures spaced `5–12 m` apart along the road).

In [ ]:
# @title Step 3: Query Horizontal SV_PANO Rosette Frames Along the 80m Corridor (with Dry-Run Check)
MAX_PANOS = 10  # ~10 captures x 6 horizontal cameras = ~60 frames along an 80m street segment

corridor_sql = rf"""
WITH nearby_frames AS (
  SELECT
    observation_id,
    capture_id,
    pano_id,
    snapshot_id,
    CAST(REGEXP_EXTRACT(observation_id, r'_(\d+)(?::|$)') AS INT64) AS cam_k,
    gcs_uri,
    capture_time,
    capture_location.latitude AS capture_lat,
    capture_location.longitude AS capture_lng,
    camera_pose.latitude AS cam_lat,
    camera_pose.longitude AS cam_lng,
    camera_pose.altitude AS cam_alt,
    camera_pose.heading AS heading,
    camera_pose.pitch AS pitch,
    camera_pose.roll AS roll,
    ST_DISTANCE(
      ST_GEOGPOINT(capture_location.longitude, capture_location.latitude),
      ST_GEOGPOINT({CORRIDOR_LNG}, {CORRIDOR_LAT})
    ) AS dist_from_center_m
  FROM `{SVI_PROJECT_ID}.{SVI_DATASET}.pano_observations_latest`
  WHERE pano_id IS NOT NULL
    AND ST_DWITHIN(
      ST_GEOGPOINT(capture_location.longitude, capture_location.latitude),
      ST_GEOGPOINT({CORRIDOR_LNG}, {CORRIDOR_LAT}),
      {CORRIDOR_RADIUS_M}
    )
)
SELECT *
FROM nearby_frames
WHERE cam_k BETWEEN 0 AND 5
ORDER BY capture_time DESC, pano_id, cam_k
"""

# 1. Perform a BigQuery dry-run first to verify bytes scanned
dry_cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry_job = bq_client.query(corridor_sql, job_config=dry_cfg)
mb_scanned = dry_job.total_bytes_processed / (1024 ** 2)
print(f"BigQuery Dry-Run: {mb_scanned:.2f} MiB to scan (~${dry_job.total_bytes_processed / (1024**4) * 6.25:.4f} USD)")

# 2. Execute the query (capped at 5 GB maximum bytes billed for safety)
run_cfg = bigquery.QueryJobConfig(maximum_bytes_billed=5 * 1024**3)
frames_df = bq_client.query(corridor_sql, job_config=run_cfg).to_dataframe()

# Keep the most recent coherent drive pass (up to MAX_PANOS captures)
top_panos = frames_df["pano_id"].drop_duplicates().head(MAX_PANOS).tolist()
frames_df = frames_df[frames_df["pano_id"].isin(top_panos)].reset_index(drop=True)

print(f"Selected {len(frames_df)} horizontal camera frames across {frames_df['pano_id'].nunique()} panoramic captures.")
frames_df[["observation_id", "pano_id", "cam_k", "capture_time", "cam_lat", "cam_lng", "heading"]].head(12)

## 4. Step 2 of Pipeline: Finding Building & Entrance Features with Gemini

For each horizontal camera frame (`3648 × 5472 px`), we ask Gemini to detect every visible street-facing building and extract **structured geometric & semantic features**:

1. **`facade_box_2d` (`[ymin, xmin, ymax, xmax]` in `0..1000`)**: Bounding box of the primary street-facing building wall.
2. **`front_door_box_2d` (`[ymin, xmin, ymax, xmax]` in `0..1000` or `None`)**: Tight bounding box around the **primary street-facing entrance door**.
   * *Why we anchor on the front door instead of the whole house box:* A whole-house bounding box changes width as the Street View car drives past (`0.49 m` ray intersection scatter). A front door is a `~1 m` wide planar feature on the front wall, which slashes 3D multi-view ray scatter **4.1× down to `0.12 m` RMS**!
3. **`left_facade_corner_u_1000` & `right_facade_corner_u_1000`**: Horizontal pixel coordinates (`0..1000`) of the left and right vertical edges of the front wall (plus `left_corner_visible` / `right_corner_visible` flags so clipped frame edges at `u < 35 px` or `u > 3613 px` are automatically down-weighted).
4. **`visible_house_number` (OCR)**: Any street address digits visibly mounted on the door, porch, or facade (used both as a **Cannot-Link guard** between adjacent homes and for address verification).

In [ ]:
# @title Step 4: Define Structured Pydantic Schema & Prompt for Building Feature Extraction
class DetectedBuildingSchema(BaseModel):
    facade_box_2d: list[int] = Field(
        description="[ymin, xmin, ymax, xmax] normalized 0..1000 of the street-facing building facade."
    )
    front_door_box_2d: list[int] | None = Field(
        default=None,
        description="[ymin, xmin, ymax, xmax] normalized 0..1000 of the primary front entrance door, or null if occluded."
    )
    garage_door_box_2d: list[int] | None = Field(
        default=None,
        description="[ymin, xmin, ymax, xmax] normalized 0..1000 of the garage door if present, else null."
    )
    left_facade_corner_u_1000: int = Field(
        description="Horizontal coordinate (0..1000) of the left vertical corner of the front facade."
    )
    right_facade_corner_u_1000: int = Field(
        description="Horizontal coordinate (0..1000) of the right vertical corner of the front facade."
    )
    left_corner_visible: bool = Field(
        description="True if the true left corner of the facade is visible and not clipped by the image border or trees."
    )
    right_corner_visible: bool = Field(
        description="True if the true right corner of the facade is visible and not clipped by the image border or trees."
    )
    building_category: str = Field(
        description="One of: DETACHED_SINGLE_FAMILY, SEMI_DETACHED_DUPLEX, TERRACED_ROWHOUSE, SMALL_MULTIFAMILY, COMMERCIAL_MIXED"
    )
    stories: float = Field(
        description="Number of above-ground storeys (e.g. 1.0, 1.5, 2.0, 3.0)."
    )
    visible_house_number: str | None = Field(
        default=None,
        description="Exact street number digits visibly printed on the house facade/porch (e.g. '530'), or null if not readable."
    )
    entrance_visibility: str = Field(
        description="One of: CLEAR_FRONT_DOOR, OCCLUDED_BY_TREE, OCCLUDED_BY_VEHICLE, GARAGE_ONLY_VISIBLE, SIDE_WALL_ONLY"
    )
    confidence: float = Field(
        description="Detection confidence from 0.0 to 1.0."
    )


class FrameBuildingDetections(BaseModel):
    buildings: list[DetectedBuildingSchema] = Field(
        default_factory=list,
        description="All distinct street-facing primary buildings clearly visible in this camera frame."
    )


HOUSE_DETECTION_PROMPT = """
Detect all distinct primary street-facing buildings (houses, residential units, or commercial buildings)
clearly visible in this Street View camera frame.

Rules for high-precision 3D photogrammetric triangulation:
1. Ignore distant background roofs > 45m away, open-air gas station canopies, backyard sheds, fences, and parked vehicles.
2. For `front_door_box_2d`, box ONLY a real pedestrian entrance door on the building wall.
   - NEVER box a second-story window, a window above a parked car, a fuel price sign, or a fence gate as `front_door_box_2d`.
   - If the front door is hidden behind trees, bushes, or a parked car, set `front_door_box_2d = null` and set `entrance_visibility` accordingly.
3. Set `left_corner_visible = false` or `right_corner_visible = false` if the building wall is cut off by the left/right edge of the image frame.
4. Read `visible_house_number` ONLY if the address digits are physically mounted on that specific building or its porch.
""".strip()


def detect_buildings_in_frames(
    df: pd.DataFrame,
    model_name: str = GEMINI_MODEL,
    max_frames: int | None = None,
) -> tuple[list[dict[str, Any]], dict[str, float]]:
    """Run Gemini structured building detection across corridor frames and track token usage."""
    client = genai.Client(vertexai=True, project=PROJECT_ID, location=VERTEX_LOCATION)
    subset = df if max_frames is None else df.head(max_frames)
    raw_detections: list[dict[str, Any]] = []
    total_in_tokens = 0
    total_out_tokens = 0

    for _, row in subset.iterrows():
        part = types.Part.from_uri(file_uri=row["gcs_uri"], mime_type="image/jpeg")
        resp = client.models.generate_content(
            model=model_name,
            contents=[part, HOUSE_DETECTION_PROMPT],
            config=types.GenerateContentConfig(
                temperature=0.0,
                response_mime_type="application/json",
                response_schema=FrameBuildingDetections,
            ),
        )
        if resp.usage_metadata:
            total_in_tokens += int(resp.usage_metadata.prompt_token_count or 0)
            total_out_tokens += int(resp.usage_metadata.candidates_token_count or 0)

        parsed = json.loads(resp.text or '{"buildings": []}')
        for b in parsed.get("buildings", []):
            raw_detections.append({
                "observation_id": row["observation_id"],
                "pano_id": row["pano_id"],
                "cam_k": int(row["cam_k"]),
                "gcs_uri": row["gcs_uri"],
                **b,
            })

    n_f = max(len(subset), 1)
    usage_summary = {
        "frames_processed": len(subset),
        "raw_building_boxes": len(raw_detections),
        "avg_input_tokens_per_frame": round(total_in_tokens / n_f, 1),
        "avg_output_tokens_per_frame": round(total_out_tokens / n_f, 1),
    }
    return raw_detections, usage_summary

print("Gemini schema and `detect_buildings_in_frames()` helper ready.")

## 5. Step 3 of Pipeline: Calibrated Camera Intrinsics & 3D Sightline Math

Each `SV_PANO` frame is captured by a **Sony IMX183 (`3648 × 5472 px`)** sensor mounted on a 6-camera horizontal ring of radius **`ROSETTE_RADIUS_M = 0.08438 m` (`8.44 cm`)**.

To convert a 2D pixel `(u, v)` into a 3D world compass sightline:
1. **8.44 cm Rosette Lever-Arm Offset (`apply_rosette_lever_arm_lla`)**: Shifts the capture center `(cam_lat, cam_lng)` outward by `8.44 cm` along camera `cam_k`'s mounting angle (`cam_k * 60°`).
2. **Radial Undistortion (`unproject_pixel_to_cam_ray`)**: Converts pixel `(u, v)` using calibrated focal length **`fx = fy = 2847.132 px`**, principal point **`(cx, cy) = (1835.6, 2736.0)`**, and first-order radial distortion **`k1 = -0.028`**.
3. **6-DoF Pose Rotation (`pixel_to_world_bearing`)**: Rotates the camera-frame ray by the 3D rotation matrix $\mathbf{R}(\text{heading}, \text{pitch}, \text{roll})$ into local **East-North-Up (`ENU`)** world coordinates to obtain compass azimuth $\theta_i \in [0^\circ, 360^\circ)$ and elevation angle $\phi_i$.

In [ ]:
# @title Step 5: Self-Contained Camera Intrinsics, 6-DoF Rotation & ENU Coordinate Transforms
R_EARTH_M = 6371008.8
ROSETTE_RADIUS_M = 0.08438098229661596
CAM_HEIGHT_ABOVE_GROUND_M = 2.45


@dataclass(frozen=True)
class CameraIntrinsics:
    width: int = 3648
    height: int = 5472
    fx: float = 2847.132359698659
    fy: float = 2847.132359698659
    cx: float = 1835.6
    cy: float = 2736.0
    k1: float = -0.028


PINHOLE_CALIBRATED = CameraIntrinsics()


def lla_to_enu(lat: float, lng: float, alt: float, ref_lat: float, ref_lng: float, ref_alt: float = 0.0) -> tuple[float, float, float]:
    """Convert WGS84 (lat, lng, alt) to local East-North-Up (ENU) meters relative to (ref_lat, ref_lng)."""
    dlat = math.radians(lat - ref_lat)
    dlng = math.radians(lng - ref_lng)
    east = dlng * R_EARTH_M * math.cos(math.radians(0.5 * (lat + ref_lat)))
    north = dlat * R_EARTH_M
    up = alt - ref_alt
    return float(east), float(north), float(up)


def enu_to_lla(east: float, north: float, up: float, ref_lat: float, ref_lng: float, ref_alt: float = 0.0) -> tuple[float, float, float]:
    """Convert local East-North-Up (ENU) meters back to WGS84 (lat, lng, alt)."""
    lat = ref_lat + math.degrees(north / R_EARTH_M)
    lng = ref_lng + math.degrees(east / (R_EARTH_M * math.cos(math.radians(0.5 * (lat + ref_lat)))))
    alt = ref_alt + up
    return float(lat), float(lng), float(alt)


def haversine_m(lat1: float, lng1: float, lat2: float, lng2: float) -> float:
    """Compute 2D horizontal distance in meters between two WGS84 coordinates."""
    e, n, _ = lla_to_enu(lat2, lng2, 0.0, lat1, lng1, 0.0)
    return float(math.hypot(e, n))


def apply_rosette_lever_arm_lla(
    cam_lat: float,
    cam_lng: float,
    cam_alt: float,
    cam_heading_deg: float,
    capture_lat: float | None = None,
    capture_lng: float | None = None,
) -> tuple[float, float, float]:
    """Return camera k's 3D optical center (+8.438 cm from capture_location without double-counting)."""
    if capture_lat is not None and capture_lng is not None:
        if haversine_m(cam_lat, cam_lng, float(capture_lat), float(capture_lng)) < 0.015:
            rad = math.radians(cam_heading_deg)
            de = ROSETTE_RADIUS_M * math.sin(rad)
            dn = ROSETTE_RADIUS_M * math.cos(rad)
            return enu_to_lla(de, dn, 0.0, cam_lat, cam_lng, cam_alt)
    return float(cam_lat), float(cam_lng), float(cam_alt)


def unproject_pixel_to_cam_ray(intr: CameraIntrinsics, u: float, v: float) -> np.ndarray:
    """Convert pixel (u, v) to a unit 3D ray [x_right, y_down, z_forward] in camera coordinates."""
    xd = (u - intr.cx) / intr.fx
    yd = (v - intr.cy) / intr.fy
    # Invert radial distortion r_d = r * (1 + k1 * r^2) via 5 Newton iterations
    x, y = xd, yd
    for _ in range(5):
        r2 = x * x + y * y
        scale = 1.0 + intr.k1 * r2
        x = xd / max(scale, 0.5)
        y = yd / max(scale, 0.5)
    vec = np.array([x, y, 1.0], dtype=float)
    return vec / float(np.linalg.norm(vec))


def pose_rotation_cam_to_enu(heading_deg: float, pitch_deg: float, roll_deg: float) -> np.ndarray:
    """Build the 3x3 rotation matrix mapping camera [right, down, forward] to world [East, North, Up]."""
    psi = math.radians(heading_deg)
    theta = math.radians(pitch_deg)
    phi = math.radians(roll_deg)

    # At heading=0, pitch=0, roll=0: cam +X is East, cam +Y is Down (-Up), cam +Z is North
    R_yaw = np.array([
        [math.cos(psi), math.sin(psi), 0.0],
        [-math.sin(psi), math.cos(psi), 0.0],
        [0.0, 0.0, 1.0],
    ], dtype=float)
    R_pitch = np.array([
        [1.0, 0.0, 0.0],
        [0.0, math.cos(theta), -math.sin(theta)],
        [0.0, math.sin(theta), math.cos(theta)],
    ], dtype=float)
    R_roll = np.array([
        [math.cos(phi), 0.0, math.sin(phi)],
        [0.0, 1.0, 0.0],
        [-math.sin(phi), 0.0, math.cos(phi)],
    ], dtype=float)
    R_base = np.array([
        [1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0],
        [0.0, -1.0, 0.0],
    ], dtype=float)
    return R_yaw @ R_base @ R_pitch @ R_roll


def pixel_to_world_bearing(
    intr: CameraIntrinsics,
    heading_deg: float,
    pitch_deg: float,
    roll_deg: float,
    u: float,
    v: float,
) -> tuple[float, float]:
    """Convert pixel (u, v) and 6-DoF camera pose into world (azimuth_deg, elevation_deg)."""
    d_cam = unproject_pixel_to_cam_ray(intr, u, v)
    R = pose_rotation_cam_to_enu(heading_deg, pitch_deg, roll_deg)
    d_enu = R @ d_cam
    az_deg = math.degrees(math.atan2(float(d_enu[0]), float(d_enu[1]))) % 360.0
    horiz = math.hypot(float(d_enu[0]), float(d_enu[1]))
    el_deg = math.degrees(math.atan2(float(d_enu[2]), max(horiz, 1e-9)))
    return float(az_deg), float(el_deg)

print("Camera intrinsics and 6-DoF pixel-to-world bearing functions initialized.")

## 6. Step 4 of Pipeline: Unsupervised 3D Sightline Clustering & House Triangulation

Now we solve the core multi-view association problem: **Given dozens of unlabeled building detections across `~60` camera frames, how do we know which boxes are looking at the same house, and how do we compute its 3D coordinates?**

### How `solve_houses_corridor()` Works

1. **Build Calibrated `HouseObservation` Rays**:
   * For each detection, we compute the compass bearing of the **front door** (`az_door_deg`), the **left/right facade corners** (`az_left_corner_deg`, `az_right_corner_deg`), and the **facade midpoint** (`az_facade_mid_deg`).
   * We also record the door's fractional position along the front wall (`alpha_door_in_facade` $\in [0.05, 0.95]$). If the door is occluded in 1 of the views (e.g., behind a tree branch), we can interpolate its bearing between the visible left and right corners!
2. **Hard Cannot-Link Guards (`_houses_cannot_link`)**:
   * **OCR Guard**: Two observations with different visible house numbers (e.g., `"530"` vs. `"534"`) can **never** be grouped into the same cluster.
   * **Same-Frame Non-Overlapping Box Guard**: Two distinct non-overlapping facade boxes in the same camera frame can **never** be linked.
3. **2-Ray Seed Triangulation & Greedy Multi-View Consensus (`<= 2.6 m` Gate)**:
   * Every pair of rays from different car captures (`2.5–65 m` apart) with **`>= 6.0°` parallax** is intersected in 2D East-North space.
   * We keep seeds at valid house distances (`3.5–48 m` from the road) that match the building's apparent pixel height (`0.35×–2.6×`), then gather **all other camera views** passing within **`<= 2.6 m` perpendicular distance**.
   * Clusters with **`>= 3` camera views** are prioritized over 2-view pairs and solved with **8-iteration Huber IRLS**.
4. **Inward Facade-Normal Shift (`+5.8 m`) to Rooftop Centroid**:
   * Triangulating the front door places the pin on the **front wall**, whereas Overture Maps and Google Places (`ChIJ`) place pins at the **rooftop center** (`~5–7 m` behind the front door).
   * We compute the 3D inward unit normal $\mathbf{n}_{\text{inward}}$ perpendicular to the facade/street and project inward by the architectural half-depth $d_{\text{offset}}$ (`~5.8 m`), outputting **both** `entrance_lat_lng` and `centroid_lat_lng`.

In [ ]:
# @title Step 6: Unsupervised 3D Sightline Graph Clustering & Facade-Plane Centroid Solver
DEFAULT_HOUSE_HALF_DEPTH_M: dict[str, float] = {
    "DETACHED_SINGLE_FAMILY": 6.2,
    "SEMI_DETACHED_DUPLEX": 5.4,
    "TERRACED_ROWHOUSE": 4.8,
    "TOWNHOUSE": 5.0,
    "SMALL_MULTIFAMILY": 6.8,
    "COMMERCIAL_MIXED": 7.5,
}


@dataclass
class HouseObservation:
    observation_id: str
    pano_id: str
    cam_k: int
    gcs_uri: str
    cam_lat: float
    cam_lng: float
    cam_alt: float
    facade_box_px: tuple[float, float, float, float]
    door_box_px: tuple[float, float, float, float] | None
    u_ref_px: float
    v_ref_px: float
    keypoint_source: str
    u_left_corner: float
    u_right_corner: float
    left_corner_visible: bool
    right_corner_visible: bool
    alpha_door_in_facade: float | None
    az_facade_mid_deg: float
    az_door_deg: float | None
    az_left_corner_deg: float
    az_right_corner_deg: float
    est_range_m: float
    weight: float
    quality_factor: float
    building_category: str
    stories: float
    visible_house_number: str | None
    entrance_visibility: str


def _norm_box_to_px(box_1000: Sequence[float] | None, w: int, h: int) -> tuple[float, float, float, float] | None:
    if not box_1000 or len(box_1000) != 4:
        return None
    y0, x0, y1, x1 = [float(v) for v in box_1000]
    px0 = float(np.clip(min(x0, x1) / 1000.0 * w, 0.0, float(w)))
    px1 = float(np.clip(max(x0, x1) / 1000.0 * w, 0.0, float(w)))
    py0 = float(np.clip(min(y0, y1) / 1000.0 * h, 0.0, float(h)))
    py1 = float(np.clip(max(y0, y1) / 1000.0 * h, 0.0, float(h)))
    if (px1 - px0) < 6.0 or (py1 - py0) < 6.0:
        return None
    return (px0, py0, px1, py1)


def build_house_observation(
    det: Mapping[str, Any],
    pose: Mapping[str, Any],
    intr: CameraIntrinsics = PINHOLE_CALIBRATED,
) -> HouseObservation | None:
    """Convert a raw per-frame Gemini building detection + camera pose into a 3D sightline observation."""
    facade_px = _norm_box_to_px(det.get("facade_box_2d") or det.get("box_2d"), intr.width, intr.height)
    if facade_px is None:
        return None
    door_px = _norm_box_to_px(det.get("front_door_box_2d") or det.get("entrance_box_2d"), intr.width, intr.height)

    fx0, fy0, fx1, fy1 = facade_px
    fh = max(fy1 - fy0, 8.0)

    u_left_1000 = det.get("left_facade_corner_u_1000")
    u_right_1000 = det.get("right_facade_corner_u_1000")
    u_left = float(np.clip(float(u_left_1000) / 1000.0 * intr.width, 0.0, float(intr.width))) if u_left_1000 is not None else fx0
    u_right = float(np.clip(float(u_right_1000) / 1000.0 * intr.width, 0.0, float(intr.width))) if u_right_1000 is not None else fx1
    if u_right <= u_left + 4.0:
        u_left, u_right = fx0, fx1

    left_vis = bool(det.get("left_corner_visible", True)) and u_left > 35.0
    right_vis = bool(det.get("right_corner_visible", True)) and u_right < (intr.width - 35.0)

    hdg = float(pose.get("heading") if pose.get("heading") is not None else pose.get("cam_hdg", 0.0))
    pitch = float(pose.get("pitch") if pose.get("pitch") is not None else pose.get("cam_pitch", 0.0))
    roll = float(pose.get("roll") if pose.get("roll") is not None else pose.get("cam_roll", 0.0))

    u_facade_mid = 0.5 * (u_left + u_right)
    v_facade_ref = fy0 + 0.78 * fh

    u_door, v_door, alpha_door = None, None, None
    kp_source = "clipped_facade_fallback"
    if door_px is not None:
        dx0, dy0, dx1, dy1 = door_px
        u_door = 0.5 * (dx0 + dx1)
        v_door = dy0 + 0.80 * (dy1 - dy0)
        kp_source = "front_door_box"
        if left_vis and right_vis and (u_right - u_left) >= 20.0:
            alpha_door = float(np.clip((u_door - u_left) / (u_right - u_left), 0.05, 0.95))
    elif left_vis and right_vis:
        kp_source = "verified_corners_midpoint"

    az_facade_mid, el_base = pixel_to_world_bearing(intr, hdg, pitch, roll, u_facade_mid, fy1)
    az_left, _ = pixel_to_world_bearing(intr, hdg, pitch, roll, u_left, v_facade_ref)
    az_right, _ = pixel_to_world_bearing(intr, hdg, pitch, roll, u_right, v_facade_ref)
    az_door = None
    if u_door is not None and v_door is not None:
        az_door, _ = pixel_to_world_bearing(intr, hdg, pitch, roll, u_door, v_door)

    stories = float(det.get("stories") or 2.0)
    typ_h_m = max(3.6, 3.1 * stories + 1.5)
    d_size = typ_h_m / max(fh / intr.fx, 0.03)
    el_adj = float(el_base) - 0.75
    d_ground = (CAM_HEIGHT_ABOVE_GROUND_M / math.tan(math.radians(-el_adj))) if (-40.0 < el_adj < -1.5 and fy1 < intr.height - 15) else None
    est_range = float(np.clip(0.5 * d_ground + 0.5 * d_size, 4.5, 45.0)) if (d_ground and 4.0 <= d_ground <= 38.0) else float(np.clip(d_size, 5.0, 45.0))

    cam_lat, cam_lng, cam_alt = apply_rosette_lever_arm_lla(
        float(pose["cam_lat"]),
        float(pose["cam_lng"]),
        float(pose.get("cam_alt") or 0.0),
        hdg,
        capture_lat=pose.get("capture_lat"),
        capture_lng=pose.get("capture_lng"),
    )

    conf = float(det.get("confidence") or 0.85)
    q_factor = (0.75 + 0.25 * conf) * (1.0 if (left_vis and right_vis) else 0.65)
    weight = float(q_factor / max(est_range, 5.0))

    hnum = det.get("visible_house_number")
    hnum_str = str(hnum).strip() if hnum not in (None, "", "null", "None") else None

    return HouseObservation(
        observation_id=str(det["observation_id"]),
        pano_id=str(det["pano_id"]),
        cam_k=int(det["cam_k"]),
        gcs_uri=str(det.get("gcs_uri") or pose.get("gcs_uri") or ""),
        cam_lat=cam_lat,
        cam_lng=cam_lng,
        cam_alt=cam_alt,
        facade_box_px=facade_px,
        door_box_px=door_px,
        u_ref_px=float(u_door if u_door is not None else u_facade_mid),
        v_ref_px=float(v_door if v_door is not None else v_facade_ref),
        keypoint_source=kp_source,
        u_left_corner=u_left,
        u_right_corner=u_right,
        left_corner_visible=left_vis,
        right_corner_visible=right_vis,
        alpha_door_in_facade=alpha_door,
        az_facade_mid_deg=float(az_facade_mid),
        az_door_deg=az_door,
        az_left_corner_deg=float(az_left),
        az_right_corner_deg=float(az_right),
        est_range_m=est_range,
        weight=weight,
        quality_factor=q_factor,
        building_category=str(det.get("building_category") or "DETACHED_SINGLE_FAMILY").upper(),
        stories=stories,
        visible_house_number=hnum_str,
        entrance_visibility=str(det.get("entrance_visibility") or ("CLEAR_FRONT_DOOR" if door_px else "OCCLUDED")),
    )


def _line_angle_deg(az1: float, az2: float) -> float:
    d = abs((az1 - az2 + 180.0) % 360.0 - 180.0)
    return min(d, 180.0 - d)


def _houses_cannot_link(o1: HouseObservation, o2: HouseObservation) -> bool:
    """Hard Cannot-Link guard: block merging distinct OCR house numbers or separate same-frame boxes."""
    if o1.visible_house_number and o2.visible_house_number and o1.visible_house_number != o2.visible_house_number:
        return True
    if o1.pano_id == o2.pano_id and o1.cam_k == o2.cam_k:
        overlap = min(o1.u_right_corner, o2.u_right_corner) - max(o1.u_left_corner, o2.u_left_corner)
        if overlap <= 0.0:
            return True
    return False


def _solve_rays_irls(
    pairs: list[tuple[HouseObservation, float]],
    ref_lat: float,
    ref_lng: float,
    min_parallax_deg: float = 5.0,
    huber_d: float = 0.55,
) -> tuple[np.ndarray | None, float, float, list[float]]:
    """Solve 2D East-North ray intersection via 8-iteration Huber IRLS."""
    if len(pairs) < 2:
        return None, 0.0, float("inf"), []
    max_p = max(
        _line_angle_deg(pairs[i][1], pairs[j][1])
        for i in range(len(pairs))
        for j in range(i + 1, len(pairs))
    )
    if max_p < min_parallax_deg:
        return None, max_p, float("inf"), []

    O_a, D_a, N_a, W_a, Q_a = [], [], [], [], []
    for o, az in pairs:
        e, n, _ = lla_to_enu(o.cam_lat, o.cam_lng, 0.0, ref_lat, ref_lng, 0.0)
        rad = math.radians(az)
        O_a.append((e, n))
        D_a.append((math.sin(rad), math.cos(rad)))
        N_a.append((math.cos(rad), -math.sin(rad)))
        W_a.append(o.weight)
        Q_a.append(o.quality_factor)

    O_m = np.array(O_a, dtype=float)
    D_m = np.array(D_a, dtype=float)
    N_m = np.array(N_a, dtype=float)
    W_m = np.array(W_a, dtype=float)
    Q_m = np.array(Q_a, dtype=float)

    try:
        xy, _, _, _ = np.linalg.lstsq(N_m * W_m[:, None], np.sum(N_m * O_m, axis=1) * W_m, rcond=None)
    except np.linalg.LinAlgError:
        return None, max_p, float("inf"), []

    for _ in range(8):
        rel = xy[None, :] - O_m
        perp = np.abs(np.sum(rel * N_m, axis=1))
        along = np.maximum(np.sum(rel * D_m, axis=1), 4.5)
        iw = np.where(perp <= huber_d, 1.0, huber_d / np.maximum(perp, 1e-6))
        eff = (np.sqrt(Q_m) / (along ** 0.70)) * np.sqrt(iw)
        try:
            nxt, _, _, _ = np.linalg.lstsq(N_m * eff[:, None], np.sum(N_m * O_m, axis=1) * eff, rcond=None)
        except np.linalg.LinAlgError:
            break
        if float(np.linalg.norm(nxt - xy)) < 1e-4:
            xy = nxt
            break
        xy = nxt

    rel_f = xy[None, :] - O_m
    along_f = np.sum(rel_f * D_m, axis=1)
    if int(np.sum((along_f >= 2.5) & (along_f <= 65.0))) < 2:
        return None, max_p, float("inf"), []
    signed_perp = np.sum(rel_f * N_m, axis=1)
    rms = float(np.sqrt(np.mean(np.square(signed_perp))))
    return xy, max_p, rms, [float(r) for r in signed_perp]


def triangulate_house_cluster(
    obs_list: Sequence[HouseObservation],
    ref_lat: float,
    ref_lng: float,
    min_parallax_deg: float = 5.0,
) -> dict[str, Any] | None:
    """Triangulate both the front entrance door and the inward-shifted rooftop centroid."""
    by_pano: dict[str, list[HouseObservation]] = {}
    for o in obs_list:
        by_pano.setdefault(o.pano_id, []).append(o)
    work = [max(grp, key=lambda g: g.weight) for grp in by_pano.values()]
    if len(work) < 2:
        return None

    xy_naive, par_naive, rms_naive, res_naive = _solve_rays_irls(
        [(o, o.az_facade_mid_deg) for o in work], ref_lat, ref_lng, min_parallax_deg, huber_d=1.10
    )
    if xy_naive is None:
        return None

    xy_left, _, _, _ = _solve_rays_irls(
        [(o, o.az_left_corner_deg) for o in work if o.left_corner_visible], ref_lat, ref_lng, min_parallax_deg, huber_d=0.75
    )
    xy_right, _, _, _ = _solve_rays_irls(
        [(o, o.az_right_corner_deg) for o in work if o.right_corner_visible], ref_lat, ref_lng, min_parallax_deg, huber_d=0.75
    )

    alphas = [o.alpha_door_in_facade for o in work if o.alpha_door_in_facade is not None]
    alpha_bar = float(np.median(alphas)) if alphas else 0.50

    door_pairs: list[tuple[HouseObservation, float]] = []
    for o in work:
        if o.az_door_deg is not None:
            door_pairs.append((o, o.az_door_deg))
        elif o.left_corner_visible and o.right_corner_visible:
            d_lr = ((o.az_right_corner_deg - o.az_left_corner_deg + 180.0) % 360.0) - 180.0
            az_interp = (o.az_left_corner_deg + alpha_bar * d_lr) % 360.0
            door_pairs.append((dataclasses.replace(o, weight=0.6 * o.weight, quality_factor=0.6 * o.quality_factor), az_interp))

    xy_door, par_door, rms_door, res_door = _solve_rays_irls(door_pairs, ref_lat, ref_lng, min_parallax_deg, huber_d=0.55)
    if xy_door is None:
        xy_door, par_door, rms_door, res_door = xy_naive.copy(), par_naive, rms_naive, res_naive

    # Estimate inward facade normal from camera trajectory baseline & triangulated corners
    cam_origins = np.array([lla_to_enu(o.cam_lat, o.cam_lng, 0.0, ref_lat, ref_lng, 0.0)[:2] for o in work], dtype=float)
    mean_cam = np.mean(cam_origins, axis=0)
    view_vec = xy_door - mean_cam
    view_unit = view_vec / max(float(np.linalg.norm(view_vec)), 1e-6)

    n_road_inward = view_unit.copy()
    if len(cam_origins) >= 2:
        _, _, vt_c = np.linalg.svd(cam_origins - mean_cam, full_matrices=False)
        t_road = vt_c[0] / max(float(np.linalg.norm(vt_c[0])), 1e-6)
        n_cand = np.array([-t_road[1], t_road[0]], dtype=float)
        if float(np.dot(n_cand, view_unit)) < 0.0:
            n_cand = -n_cand
        n_road_inward = n_cand

    est_width_m = None
    n_inward = n_road_inward
    if xy_left is not None and xy_right is not None:
        seg = xy_right - xy_left
        seg_len = float(np.linalg.norm(seg))
        if 3.2 <= seg_len <= 28.0:
            est_width_m = seg_len
            t_seg = seg / seg_len
            n_c = np.array([-t_seg[1], t_seg[0]], dtype=float)
            if float(np.dot(n_c, view_unit)) < 0.0:
                n_c = -n_c
            if float(np.dot(n_c, n_road_inward)) >= math.cos(math.radians(32.0)):
                n_inward = 0.55 * n_c + 0.45 * n_road_inward
                n_inward = n_inward / max(float(np.linalg.norm(n_inward)), 1e-9)

    if est_width_m is None:
        w_ests = [
            2.0 * float(np.linalg.norm(xy_door - orig)) * math.tan(0.5 * math.radians(_line_angle_deg(o.az_right_corner_deg, o.az_left_corner_deg)))
            for o, orig in zip(work, cam_origins)
        ]
        est_width_m = float(np.clip(np.median(w_ests) if w_ests else 9.5, 4.0, 22.0))

    t_facade = np.array([n_inward[1], -n_inward[0]], dtype=float)
    if (view_unit[0] * t_facade[1] - view_unit[1] * t_facade[0]) > 0.0:
        t_facade = -t_facade

    lateral_shift_m = float(np.clip((0.50 - alpha_bar) * est_width_m, -4.5, 4.5))
    xy_door_shifted = xy_door + lateral_shift_m * t_facade
    xy_facade_center = 0.65 * xy_door_shifted + 0.35 * xy_naive

    cats = [o.building_category for o in work if o.building_category]
    best_cat = max(set(cats), key=cats.count) if cats else "DETACHED_SINGLE_FAMILY"
    base_depth_m = float(DEFAULT_HOUSE_HALF_DEPTH_M.get(best_cat, 5.8))
    d_offset_m = float(np.clip(base_depth_m + np.clip(0.15 * (est_width_m - 9.0), -1.0, 1.5), 2.5, 11.0))

    xy_cent = xy_facade_center + d_offset_m * n_inward
    ent_lat, ent_lng, _ = enu_to_lla(float(xy_door[0]), float(xy_door[1]), 0.0, ref_lat, ref_lng, 0.0)
    cent_lat, cent_lng, _ = enu_to_lla(float(xy_cent[0]), float(xy_cent[1]), 0.0, ref_lat, ref_lng, 0.0)

    hnums = [o.visible_house_number for o in work if o.visible_house_number]
    best_hnum = max(set(hnums), key=hnums.count) if hnums else None
    n_door_views = sum(1 for o in work if o.door_box_px is not None)

    return {
        "entrance_lat": round(ent_lat, 7),
        "entrance_lng": round(ent_lng, 7),
        "centroid_lat": round(cent_lat, 7),
        "centroid_lng": round(cent_lng, 7),
        "n_views": len(work),
        "n_door_views": n_door_views,
        "max_parallax_deg": round(par_door, 2),
        "ray_rms_m": round(rms_door, 3),
        "est_facade_width_m": round(est_width_m, 2),
        "depth_offset_m": round(d_offset_m, 2),
        "building_category": best_cat,
        "visible_house_number": best_hnum,
        "observations": work,
    }


def solve_houses_corridor(
    frames_df: pd.DataFrame,
    raw_detections: Sequence[Mapping[str, Any]],
    min_parallax_deg: float = 6.0,
    cluster_perp_gate_m: float = 2.6,
    dedup_dist_m: float = 4.2,
) -> list[dict[str, Any]]:
    """Group unlabeled building detections across frames into physical houses and triangulate each one."""
    pose_by_obs = {row["observation_id"]: row.to_dict() for _, row in frames_df.iterrows()}
    obs_all: list[HouseObservation] = []
    for det in raw_detections:
        pose = pose_by_obs.get(str(det.get("observation_id")))
        if pose is not None:
            ho = build_house_observation(det, pose)
            if ho is not None:
                obs_all.append(ho)

    if len(obs_all) < 2:
        return []

    ref_lat = float(frames_df["cam_lat"].mean())
    ref_lng = float(frames_df["cam_lng"].mean())

    # Step 1: Same-pano cross-seam deduplication (< 6 deg bearing difference)
    by_pano: dict[str, list[HouseObservation]] = {}
    for o in obs_all:
        by_pano.setdefault(o.pano_id, []).append(o)
    dedup_obs: list[HouseObservation] = []
    for rs in by_pano.values():
        used: list[HouseObservation] = []
        for r in sorted(rs, key=lambda x: -x.weight):
            az_r = r.az_door_deg if r.az_door_deg is not None else r.az_facade_mid_deg
            if not any(
                _line_angle_deg(az_r, u.az_door_deg if u.az_door_deg is not None else u.az_facade_mid_deg) < 6.0
                and u.cam_k != r.cam_k
                and not _houses_cannot_link(r, u)
                for u in used
            ):
                used.append(r)
        dedup_obs.extend(used)

    n = len(dedup_obs)
    if n < 2:
        return []

    O = np.zeros((n, 2), dtype=float)
    D = np.zeros((n, 2), dtype=float)
    N = np.zeros((n, 2), dtype=float)
    W = np.array([o.weight for o in dedup_obs], dtype=float)
    P = [o.pano_id for o in dedup_obs]
    AZ = [o.az_door_deg if o.az_door_deg is not None else o.az_facade_mid_deg for o in dedup_obs]
    M = np.array([o.est_range_m for o in dedup_obs], dtype=float)

    for i, o in enumerate(dedup_obs):
        e, nr, _ = lla_to_enu(o.cam_lat, o.cam_lng, 0.0, ref_lat, ref_lng, 0.0)
        rad = math.radians(AZ[i])
        O[i] = (e, nr)
        D[i] = (math.sin(rad), math.cos(rad))
        N[i] = (math.cos(rad), -math.sin(rad))

    # Step 2: 2-Ray Seed Intersections
    cands = []
    for i in range(n):
        for j in range(i + 1, n):
            if P[i] == P[j] or _houses_cannot_link(dedup_obs[i], dedup_obs[j]):
                continue
            sep = float(np.linalg.norm(O[i] - O[j]))
            if not (2.5 <= sep <= 65.0) or _line_angle_deg(AZ[i], AZ[j]) < min_parallax_deg:
                continue
            try:
                pt = np.linalg.solve(np.vstack([N[i], N[j]]), np.array([N[i] @ O[i], N[j] @ O[j]]))
            except np.linalg.LinAlgError:
                continue
            ti, tj = float((pt - O[i]) @ D[i]), float((pt - O[j]) @ D[j])
            if 3.5 <= ti <= 48.0 and 3.5 <= tj <= 48.0 and 0.35 <= ti / M[i] <= 2.6 and 0.35 <= tj / M[j] <= 2.6:
                cands.append((pt, i, j))

    # Step 3: Greedy Multi-View Inlier Consensus (<= 2.6 m gate, prioritizing >= 3 views)
    used_mask = np.zeros(n, dtype=bool)
    located: list[dict[str, Any]] = []
    while True:
        best = None
        for pt, i, j in cands:
            if used_mask[i] or used_mask[j]:
                continue
            rel = pt[None, :] - O
            along = np.sum(rel * D, axis=1)
            perp = np.abs(np.sum(rel * N, axis=1))
            q = along / M
            ok = (~used_mask) & (along >= 3.5) & (along <= 48.0) & (perp <= cluster_perp_gate_m) & (q >= 0.35) & (q <= 2.6)
            bp: dict[str, int] = {}
            for kk in np.nonzero(ok)[0]:
                k = int(kk)
                if _houses_cannot_link(dedup_obs[i], dedup_obs[k]) or _houses_cannot_link(dedup_obs[j], dedup_obs[k]):
                    continue
                pk = P[k]
                if pk not in bp or perp[k] < perp[bp[pk]]:
                    bp[pk] = k
            inl = list(bp.values())
            if len(inl) < 2:
                continue
            tier_p = 1 if len(inl) >= 3 else 0
            sc = float(np.sum(W[inl] * (1.0 - 0.35 * perp[inl] / cluster_perp_gate_m)))
            if best is None or (tier_p, sc) > best[0]:
                best = ((tier_p, sc), inl)
        if best is None:
            break
        _, inl = best
        tri = triangulate_house_cluster([dedup_obs[k] for k in inl], ref_lat, ref_lng, min_parallax_deg)
        for k in inl:
            used_mask[k] = True
        if tri is not None and tri["ray_rms_m"] <= 1.85:
            located.append(tri)

    # Step 4: Spatial Deduplication (preserving rowhouses with distinct house numbers)
    kept: list[dict[str, Any]] = []
    for h in sorted(located, key=lambda x: (-x["n_views"], x["ray_rms_m"])):
        if any(
            not (h.get("visible_house_number") and k.get("visible_house_number") and h["visible_house_number"] != k["visible_house_number"])
            and haversine_m(h["entrance_lat"], h["entrance_lng"], k["entrance_lat"], k["entrance_lng"]) < dedup_dist_m
            for k in kept
        ):
            continue
        h["cluster_id"] = f"H{len(kept)+1:02d}"
        kept.append(h)
    return kept

print("`solve_houses_corridor()` 3D sightline clustering and triangulation engine defined.")

## 7. Automated Quality Gates & Failure-Mode Flags

Before handing triangulated coordinates to downstream business systems, we evaluate **3 automated quality gates** on every triangulated house cluster (derived from our forensic outlier analysis):

1. **`PASS_RAY_RMS` (`ray_rms_m <= 0.50 m`)**:
   * **What it catches:** Heavy tree-canopy occlusion (where the keypoint jumps across branches/windows, inflating Ray RMS to `~0.98 m`) or open-air gas station canopies (`~0.92 m` RMS).
2. **`PASS_DOOR_ANCHOR` (`n_door_views >= 1`)**:
   * **What it catches:** Blank window-slit side walls (`0/2` doors detected, `clipped_facade_fallback`) where the entrance faces a narrow side alley rather than the street.
3. **`PASS_MULTIVIEW_REDUNDANCY` (`n_views >= 3` or matching multi-view OCR)**:
   * **What it catches:** The **2-View `0.00 m` RMS Degeneracy Trap** (any 2 non-parallel 2D lines intersect at a single point with `0.00 m` residual, so a 2-view cluster without a 3rd ray cannot self-reject a cross-door swap on repetitive rowhouses).

In [ ]:
# @title Step 7: Apply Automated Quality Gates & Run Corridor Discovery
def evaluate_quality_gates(house: Mapping[str, Any]) -> dict[str, Any]:
    """Attach automated quality-gate flags and diagnostic status to a triangulated house."""
    pass_rms = float(house["ray_rms_m"]) <= 0.50
    pass_door = int(house["n_door_views"]) >= 1
    pass_redundancy = int(house["n_views"]) >= 3 or (house.get("visible_house_number") is not None)

    flags = []
    if not pass_rms:
        flags.append("HIGH_RAY_RMS_SCATTER (>0.50m: possible tree/canopy occlusion)")
    if not pass_door:
        flags.append("NO_DIRECT_DOOR_BOX (0 door views: side-wall or heavy occlusion fallback)")
    if not pass_redundancy:
        flags.append("TWO_VIEW_UNVERIFIED (2 views without OCR: 0.00m RMS swap risk)")

    return {
        "quality_tier": "HIGH_CONFIDENCE" if (pass_rms and pass_door and pass_redundancy) else "REVIEW_RECOMMENDED",
        "pass_ray_rms": pass_rms,
        "pass_door_anchor": pass_door,
        "pass_multiview_redundancy": pass_redundancy,
        "diagnostic_flags": "; ".join(flags) if flags else "CLEAN_3D_CONVERGENCE",
    }


# Run Gemini feature detection across the corridor frames (or load cached detections if re-running)
raw_detections, token_stats = detect_buildings_in_frames(frames_df)
print("Token & Detection Summary:", json.dumps(token_stats, indent=2))

# Cluster sightlines in 3D and triangulate all houses along the corridor
triangulated_houses = solve_houses_corridor(frames_df, raw_detections)
for h in triangulated_houses:
    h.update(evaluate_quality_gates(h))

houses_summary_df = pd.DataFrame([
    {
        "cluster_id": h["cluster_id"],
        "n_views": h["n_views"],
        "n_door_views": h["n_door_views"],
        "ocr_number": h["visible_house_number"],
        "entrance_lat": h["entrance_lat"],
        "entrance_lng": h["entrance_lng"],
        "centroid_lat": h["centroid_lat"],
        "centroid_lng": h["centroid_lng"],
        "ray_rms_m": h["ray_rms_m"],
        "parallax_deg": h["max_parallax_deg"],
        "facade_width_m": h["est_facade_width_m"],
        "depth_shift_m": h["depth_offset_m"],
        "quality_tier": h["quality_tier"],
        "diagnostic_flags": h["diagnostic_flags"],
    }
    for h in triangulated_houses
])

print(f"\nTriangulated {len(houses_summary_df)} unique buildings along the corridor:")
houses_summary_df

## 8. Step 5 of Pipeline: Post-Hoc Ground-Truth Verification (Overture Maps + Google Places `ChIJ`)

Now that the model has blindly produced a table of predicted `(centroid_lat, centroid_lng)` coordinates, how do we verify accuracy and identify the official street address of each discovered house?

1. **Query Overture Building Footprints (`bigquery-public-data.overture_maps.building`)**:
   * Retrieves all ground-truth roof polygons (`id` = Overture `GERS UUID`) within `45 m` of the street corridor.
2. **1-to-1 Hungarian Bipartite Matching (`scipy.optimize.linear_sum_assignment`)**:
   * Pairs each predicted rooftop centroid with at most one ground-truth Overture roof centroid within match radius **`R = 10.0 m`** (and `R = 5.0 m`).
3. **Point-in-Polygon Containment (`inside_footprint_polygon`)**:
   * Checks whether our predicted rooftop coordinate `(centroid_lat, centroid_lng)` lands **strictly inside** the matched Overture building roof polygon (`88.8%` across our 5-city benchmark).
4. **Google Places API v1 (`places:searchText`) Verification**:
   * Queries Google Places API v1 around each matched building to retrieve the official **`ChIJ...` Place ID**, `formattedAddress`, and rooftop pin coordinate, then measures the exact distance in meters (`dist_to_google_places_m`).

### How to Interpret the Verification Metrics
* **Why Strict Hungarian Recall (`46.0%` 5-city overall, `82.5%` in open US suburbs) is lower than In-Roof Hit Rate (`88.8%`)**: Overture Maps includes every roof polygon within `38–45 m` of the road—including backyard sheds, detached rear garages, and buildings hidden behind dense fences/trees that are physically invisible from the street camera.
* **Why Strict Hungarian Precision (`44.8%`) is lower than Nearest-Building Snap Rate (`91.8%`)**: Under 1-to-1 matching, if Gemini detects 2 doors on 1 wide multi-unit building polygon, the 2nd door is counted as an "unmatched prediction" in strict 1-to-1 F1 even though it lands inside a real building roof.
* **Google Places Hit Rate (`94.4% <= 10 m`, `3.67 m` median error)**: Measures how accurately a verified residential house is localized relative to its official Google Maps `ChIJ...` pin.

In [ ]:
# @title Step 8: Verify Triangulated Houses Against Overture Roofs & Google Places API v1
overture_sql = f"""
SELECT
  id AS overture_gers_id,
  ST_ASTEXT(geometry) AS footprint_wkt,
  ST_Y(ST_CENTROID(geometry)) AS gt_centroid_lat,
  ST_X(ST_CENTROID(geometry)) AS gt_centroid_lng
FROM `bigquery-public-data.overture_maps.building`
WHERE ST_DWITHIN(
  geometry,
  ST_GEOGPOINT({CORRIDOR_LNG}, {CORRIDOR_LAT}),
  {CORRIDOR_RADIUS_M + 20.0}
)
"""

gt_df = bq_client.query(overture_sql, job_config=bigquery.QueryJobConfig(maximum_bytes_billed=10 * 1024**3)).to_dataframe()
print(f"Loaded {len(gt_df)} Overture ground-truth building footprints in corridor.")


def lookup_google_place_v1(lat: float, lng: float, house_num: str | None = None) -> dict[str, Any] | None:
    """Query Google Places API v1 near (lat, lng) to retrieve official ChIJ Place ID and rooftop coordinate."""
    if not ENABLE_GOOGLE_PLACES_VERIFICATION:
        return None
    url = "https://places.googleapis.com/v1/places:searchText"
    q_str = f"{house_num} W Myrtle St, Lakeland, FL" if house_num else f"{lat:.6f}, {lng:.6f}"
    headers = {
        "Content-Type": "application/json",
        "X-Goog-User-Project": PROJECT_ID,
        "X-Goog-FieldMask": "places.id,places.formattedAddress,places.location,places.types",
    }
    body = {
        "textQuery": q_str,
        "locationBias": {
            "circle": {"center": {"latitude": lat, "longitude": lng}, "radius": 25.0}
        },
    }
    try:
        r = authed_http.post(url, headers=headers, json=body, timeout=15)
        if r.status_code != 200:
            return None
        places = r.json().get("places", [])
        if not places:
            return None
        best_p = min(
            places,
            key=lambda p: haversine_m(lat, lng, p["location"]["latitude"], p["location"]["longitude"]),
        )
        plat = float(best_p["location"]["latitude"])
        plng = float(best_p["location"]["longitude"])
        if haversine_m(lat, lng, plat, plng) > 25.0:
            return None
        return {
            "google_place_id": best_p["id"],
            "formatted_address": best_p.get("formattedAddress"),
            "place_lat": plat,
            "place_lng": plng,
        }
    except Exception:
        return None


# Run 1-to-1 Hungarian Bipartite Matching between predicted centroids and Overture roof centroids
MATCH_RADIUS_M = 10.0
n_pred = len(triangulated_houses)
n_gt = len(gt_df)

cost_matrix = np.full((n_pred, n_gt), 1e6, dtype=float)
gt_polys = [wkt.loads(w) for w in gt_df["footprint_wkt"]]

for i, h in enumerate(triangulated_houses):
    for j, row in gt_df.iterrows():
        d_m = haversine_m(h["centroid_lat"], h["centroid_lng"], float(row["gt_centroid_lat"]), float(row["gt_centroid_lng"]))
        if d_m <= MATCH_RADIUS_M:
            cost_matrix[i, j] = d_m

row_ind, col_ind = linear_sum_assignment(cost_matrix)
matched_pairs = {int(r): int(c) for r, c in zip(row_ind, col_ind) if cost_matrix[r, c] <= MATCH_RADIUS_M}

verification_rows = []
for i, h in enumerate(triangulated_houses):
    pt = Point(h["centroid_lng"], h["centroid_lat"])
    # Check if strictly inside ANY real Overture building polygon
    inside_any_roof = any(poly.contains(pt) or poly.touches(pt) for poly in gt_polys)
    j = matched_pairs.get(i)
    if j is not None:
        gt_row = gt_df.iloc[j]
        roof_err_m = round(float(cost_matrix[i, j]), 2)
        inside_matched = bool(gt_polys[j].contains(pt) or gt_polys[j].touches(pt))
        place_info = lookup_google_place_v1(float(gt_row["gt_centroid_lat"]), float(gt_row["gt_centroid_lng"]), h.get("visible_house_number"))
        places_err_m = (
            round(haversine_m(h["centroid_lat"], h["centroid_lng"], place_info["place_lat"], place_info["place_lng"]), 2)
            if place_info else None
        )
        verification_rows.append({
            "cluster_id": h["cluster_id"],
            "matched_gers_id": gt_row["overture_gers_id"],
            "google_place_id": place_info["google_place_id"] if place_info else None,
            "verified_address": place_info["formatted_address"] if place_info else None,
            "n_views": h["n_views"],
            "ocr": h["visible_house_number"],
            "ray_rms_m": h["ray_rms_m"],
            "roof_centroid_err_m": roof_err_m,
            "google_places_err_m": places_err_m,
            "inside_roof_polygon": inside_matched or inside_any_roof,
            "quality_tier": h["quality_tier"],
        })
    else:
        verification_rows.append({
            "cluster_id": h["cluster_id"],
            "matched_gers_id": None,
            "google_place_id": None,
            "verified_address": None,
            "n_views": h["n_views"],
            "ocr": h["visible_house_number"],
            "ray_rms_m": h["ray_rms_m"],
            "roof_centroid_err_m": None,
            "google_places_err_m": None,
            "inside_roof_polygon": inside_any_roof,
            "quality_tier": h["quality_tier"],
        })

verif_df = pd.DataFrame(verification_rows)
tp = len(matched_pairs)
prec = tp / max(n_pred, 1)
rec = tp / max(n_gt, 1)
f1 = (2 * prec * rec / max(prec + rec, 1e-9)) if (prec + rec) > 0 else 0.0
in_roof_pct = 100.0 * verif_df["inside_roof_polygon"].mean() if len(verif_df) else 0.0
med_roof_err = verif_df["roof_centroid_err_m"].dropna().median()
med_places_err = verif_df["google_places_err_m"].dropna().median()

print(
    f"Corridor Verification Scorecard (@R={MATCH_RADIUS_M}m):\n"
    f"  • Matched Houses (TP): {tp} / {n_pred} predicted ({n_gt} Overture roofs in buffer)\n"
    f"  • Strict 1-to-1 Precision: {100*prec:.1f}% | Recall: {100*rec:.1f}% | F1: {100*f1:.1f}%\n"
    f"  • Strictly Inside Building Roof Polygon: {in_roof_pct:.1f}%\n"
    f"  • Median Error vs. Overture Roof Centroid: {med_roof_err:.2f} m\n"
    f"  • Median Error vs. Official Google Places (ChIJ) Pin: {med_places_err:.2f} m"
    if pd.notna(med_places_err) else
    f"Corridor Verification Scorecard (@R={MATCH_RADIUS_M}m): TP={tp}, F1={100*f1:.1f}%, Med Roof Err={med_roof_err:.2f}m"
)
verif_df

## 9. Visual Verification: Multi-View `SV_PANO` Overlays & 2D Sightline Map

Finally, we inspect our best-verified house cluster side-by-side:
1. **Top Row**: Up to 3 `SV_PANO` camera frames that clustered into the same house, showing the **Facade Bounding Box (cyan dashed)**, **Front Door Bounding Box (green solid)**, **Triangulated Door Keypoint (red dot)**, and **Vertical Sightline (yellow dash-dot)**.
2. **Bottom Plot**: Overhead 2D East-North verification map showing the **Street View Camera Positions (blue dots)**, **Intersecting 3D Sightlines (cyan lines)**, **Triangulated Front Door (orange dot)**, **`+5.8 m` Inward Normal Shift (red arrow)**, **Predicted Rooftop Centroid (red star)**, and **Overture Ground-Truth Roof Polygon (green polygon)**.

In [ ]:
# @title Step 9: Render Multi-View SV_PANO Overlays & Overhead 2D Sightline Map
def download_image_in_memory(gcs_uri: str) -> Image.Image:
    """Stream a GCS image into memory using AuthorizedSession (zero local file persistence)."""
    path_str = gcs_uri.replace("gs://", "")
    bucket_name, obj_name = path_str.split("/", 1)
    url = f"https://storage.googleapis.com/storage/v1/b/{bucket_name}/o/{urllib.parse.quote(obj_name, safe='')}?alt=media"
    resp = authed_http.get(url, timeout=60)
    resp.raise_for_status()
    return Image.open(io.BytesIO(resp.content))


if triangulated_houses:
    # Pick the highest-view matched cluster for visual inspection
    best_idx = max(
        range(len(triangulated_houses)),
        key=lambda i: (i in matched_pairs, triangulated_houses[i]["n_views"], -triangulated_houses[i]["ray_rms_m"]),
    )
    target_h = triangulated_houses[best_idx]
    target_v = verif_df.iloc[best_idx]
    views = target_h["observations"][:3]

    fig = plt.figure(figsize=(15, 8.5), dpi=130, facecolor="#FFFFFF")
    gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 1.15], hspace=0.28, wspace=0.15)

    for col_i, v in enumerate(views):
        ax = fig.add_subplot(gs[0, col_i])
        img = download_image_in_memory(v.gcs_uri)
        W, H = img.size
        fbox = v.facade_box_px
        dbox = v.door_box_px
        u_ref, v_ref = v.u_ref_px, v.v_ref_px

        cx, cy = 0.5 * (fbox[0] + fbox[2]), 0.5 * (fbox[1] + fbox[3])
        bw = max(fbox[2] - fbox[0], 1100) * 1.6
        bh = bw / 1.45
        x1, x2 = max(0, int(cx - bw / 2)), min(W, int(cx + bw / 2))
        y1, y2 = max(0, int(cy - bh / 2)), min(H, int(cy + bh / 2))

        ax.imshow(img.crop((x1, y1, x2, y2)))
        ax.add_patch(mpatches.Rectangle(
            (fbox[0] - x1, fbox[1] - y1), fbox[2] - fbox[0], fbox[3] - fbox[1],
            fill=False, edgecolor="#38BDF8", linewidth=2.0, linestyle="--", label="Facade Box"
        ))
        if dbox:
            ax.add_patch(mpatches.Rectangle(
                (dbox[0] - x1, dbox[1] - y1), dbox[2] - dbox[0], dbox[3] - dbox[1],
                fill=False, edgecolor="#22C55E", linewidth=2.4, label="Front Door Box"
            ))
        ax.axvline(u_ref - x1, color="#FACC15", linestyle="-.", linewidth=1.6)
        ax.scatter([u_ref - x1], [v_ref - y1], s=70, color="#EF4444", edgecolor="#FFFFFF", linewidth=1.5, zorder=9)
        ax.set_xticks([])
        ax.set_yticks([])
        az_use = v.az_door_deg if v.az_door_deg is not None else v.az_facade_mid_deg
        ax.set_title(
            f"View #{col_i+1} (cam_{v.cam_k} • {az_use:.1f}°)\nKP: {v.keypoint_source} • OCR: {v.visible_house_number}",
            fontsize=9.5, fontweight="bold"
        )
        ax.text(
            0.98, 0.03, "Imagery © Google", transform=ax.transAxes,
            ha="right", va="bottom", fontsize=7.5, color="#FFFFFF",
            bbox={"facecolor": "#000000", "alpha": 0.55, "pad": 2, "edgecolor": "none"}
        )

    # Bottom Panel: Overhead 2D East-North Sightline & Roof Polygon Plot
    ax_map = fig.add_subplot(gs[1, :])
    ref_lat, ref_lng = target_h["entrance_lat"], target_h["entrance_lng"]
    c_e, c_n, _ = lla_to_enu(target_h["centroid_lat"], target_h["centroid_lng"], 0.0, ref_lat, ref_lng, 0.0)

    # Plot matched Overture footprint polygon if available
    if best_idx in matched_pairs:
        poly = gt_polys[matched_pairs[best_idx]]
        ext_coords = [lla_to_enu(lat, lng, 0.0, ref_lat, ref_lng, 0.0)[:2] for lng, lat in poly.exterior.coords]
        ax_map.add_patch(mpatches.Polygon(
            ext_coords, closed=True, facecolor="#DCFCE7", edgecolor="#16A34A", linewidth=2.2, alpha=0.7, label="Overture Roof Polygon (GT)"
        ))

    for v in target_h["observations"]:
        cam_e, cam_n, _ = lla_to_enu(v.cam_lat, v.cam_lng, 0.0, ref_lat, ref_lng, 0.0)
        ax_map.plot([cam_e, 0.0], [cam_n, 0.0], color="#0284C7", linewidth=1.8, alpha=0.85)
        ax_map.scatter([cam_e], [cam_n], s=60, color="#1D4ED8", edgecolor="#FFFFFF", linewidth=1.4, zorder=6)

    ax_map.annotate(
        "", xy=(c_e, c_n), xytext=(0.0, 0.0),
        arrowprops={"arrowstyle": "-|>", "color": "#DC2626", "lw": 2.4, "mutation_scale": 15}
    )
    ax_map.scatter([0.0], [0.0], s=110, color="#F59E0B", edgecolor="#FFFFFF", linewidth=1.6, zorder=8, label="Triangulated Front Door (0, 0)")
    ax_map.scatter([c_e], [c_n], s=180, marker="*", color="#EF4444", edgecolor="#FFFFFF", linewidth=1.4, zorder=9, label=f"Predicted Roof Centroid (+{target_h['depth_offset_m']}m shift)")

    addr_lbl = target_v.get("verified_address") or f"Cluster {target_h['cluster_id']}"
    ax_map.set_title(
        f"Overhead 2D Sightline Convergence: {addr_lbl} | {target_h['n_views']} Views | Ray RMS = {target_h['ray_rms_m']:.2f} m | Roof Err = {target_v.get('roof_centroid_err_m')} m",
        fontsize=10.5, fontweight="bold"
    )
    ax_map.set_xlabel("East Offset from Triangulated Entrance (meters)")
    ax_map.set_ylabel("North Offset from Triangulated Entrance (meters)")
    ax_map.axis("equal")
    ax_map.grid(True, linestyle="--", alpha=0.35)
    ax_map.legend(loc="upper right", fontsize=8.5)
    plt.show()